# Project - Airline AI Assistant

In [1]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [2]:
# Initialization

load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
MODEL = "gpt-5.4-mini"
openai = OpenAI()



OpenAI API Key exists and begins sk-proj-


In [30]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


## Tools

In [3]:
import sqlite3


In [4]:
DB = "prices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY,country TEXT, price REAL,ETA REAL )')
    conn.commit()

In [7]:
# from pathlib import Path
# Path(DB).unlink(missing_ok=True)

In [ ]:
# import pandas as pd
# df = pd.read_csv('New_Delhi_Capital_Flights_Medians.csv')
# df.head()

# for row in df.itertuples(index=False, name=None):
#     set_ticket_price(*row)

In [8]:
# for i in df.iloc[0]:
#     print(i)

In [32]:
def to_hours(eta):
    if isinstance(eta, str):
        hours, minutes, seconds = map(float, eta.split(":"))
        return hours + minutes / 60 + seconds / 3600
    return float(eta) 

to_hours("01:47:30")

def set_ticket_price(city, country, price, eta):
    with sqlite3.connect(DB) as conn:
        conn.execute("""
            INSERT INTO prices (city, country, price, ETA)
            VALUES (?, ?, ?, ?)
            ON CONFLICT(city) DO UPDATE SET
                country = excluded.country,
                price = excluded.price,
                ETA = excluded.ETA
        """, (
            city.lower(),
            country.lower(),
            float(price),
            to_hours(eta)
        ))

def get_ticket_price(city):
    print(f"DATABASE TOOL 1 CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {city} is Rs {result[0]}" if result else "No price data available for this city"

def get_ticket_eta(city):
    print(f"DATABASE TOOL 2 CALLED: Getting eta for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT ETA FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"Time taken to {city} is {result[0]} hours" if result else "No time data available for this city"


import sqlite3
import pandas as pd

with sqlite3.connect(DB) as conn:
    df1 = pd.read_sql_query("SELECT * FROM prices", conn)

display(df1)  # In a Jupyter notebook

,city,country,price,ETA
0,kathmandu,nepal,11000.0,1.791667
1,dhaka,bangladesh,13000.0,2.625000
2,malé,maldives,18500.0,6.000000
3,singapore,singapore,20000.0,5.750000
4,kuala lumpur,malaysia,17000.0,5.750000
5,bangkok,thailand,19000.0,4.500000
6,hanoi,vietnam,19000.0,4.500000
7,phnom penh,cambodia,26500.0,8.500000
8,vientiane,laos,28000.0,11.000000
9,tokyo,japan,44000.0,8.750000


In [33]:
get_ticket_price('Paris') 

DATABASE TOOL 1 CALLED: Getting price for Paris


'Ticket price to Paris is Rs 42500.0'

In [25]:
get_ticket_eta('Paris') 

DATABASE TOOL 2 CALLED: Getting eta for Paris


'Time taken to Paris is 13.0 hours'

In [34]:
# There's a particular dictionary structure that's required to describe our function:

price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

eta_function = {
    "name": "get_ticket_eta",
    "description": "Get the estimated time of arrival to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

tools = [{"type": "function", "function": price_function},{"type": "function", "function": eta_function}]

In [35]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
        elif tool_call.function.name == "get_ticket_eta":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            eta_details = get_ticket_eta(city)
            responses.append({
                "role": "tool",
                "content": eta_details,
                "tool_call_id": tool_call.id
            })
    return responses

In [36]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages)
    
    return response.choices[0].message.content

In [37]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


DATABASE TOOL 1 CALLED: Getting price for Paris
DATABASE TOOL 2 CALLED: Getting eta for Paris
DATABASE TOOL 1 CALLED: Getting price for London
DATABASE TOOL 2 CALLED: Getting eta for Dhaka
